In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep
import pandas as pd
import csv
from bs4 import BeautifulSoup
from lxml import etree
import cssselect
import requests

In [ ]:
titles = []
links=[]
prices = []
lin=[]
featuers=[]
content=[]
basic=[]
Engine=[]
Exterior_color=[]
Interior_color=[]
Drivetrain=[]
MPG=[]
Fuel_type=[]
Transmission=[]
Stock=[]
Mileage=[]

In [ ]:
number_of_pages = 2
number_of_results_on_page = 50

In [ ]:
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument('--ignore-certificate-errors')
options.add_argument('--ignore-ssl-errors')
options.add_argument('--no-sandbox')

driver = webdriver.Chrome(options=options)
driver.get('https://www.cars.com/shopping/results/?dealer_id=&keyword=&list_price_max=&list_price_min=&makes[]=&maximum_distance=all&mileage_max=&monthly_payment=&page_size='+str(number_of_results_on_page)+'&sort=best_match_desc&stock_type=all&year_max=&year_min=&zip=')
for i in range(number_of_pages):
    print('Scraping page', i+1)
    title = driver.find_elements(By.CLASS_NAME,'title')
    price = driver.find_elements(By.CLASS_NAME,'primary-price')
    link = driver.find_elements(By.TAG_NAME,'a')
    for p in price:
        prices.append(p.text)
    for t in title:
        titles.append(t.text)
    for l in link:
        links.append(l.get_attribute("href"))
    next_button =WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="next_paginate"]')))
    next_button.click()
#driver.quit()

In [ ]:
x = 'vehicledetail'
for i in links:
    if i is None:
         links.remove(i)

for i in links:
    lin = i.split('/')
    if x in lin:
        content.append(i)
print(links)

In [ ]:
VIN=[]
for i in content:
    webpage = requests.get(i)
    soup = BeautifulSoup(webpage.content, "html.parser")
    dl=soup.find('dl',class_='fancy-description-list')
    list1=[]
    list2=[]
    list3=[]
    list4=[]
    #print(dl.find_all('dt'))
    for d in dl.find_all('dt'):
        list1.append(d.text.split('>'))
    for d in dl.find_all('dd'):
        list2.append(d.text.split('>'))
    for i in list1:
        list3.append(i[0])
    for i in list2:
        list4.append(i[0])
        
    for i in range(len(list3)):
        if str('VIN')in list3[i]:
            index1 = list3.index('VIN')
            s=str(list4[index1])
            if s in VIN:
                print('NO')
            if s not in VIN:
                index = list3.index('Engine')
                Engine.append(list4[index])
                index2 = list3.index('Exterior color')
                Exterior_color.append(list4[index2])
                index3 = list3.index('Interior color')
                Interior_color.append(list4[index3])
                index4 = list3.index('Drivetrain')
                Drivetrain.append(list4[index4])
                if str('MPG') in list3:
                    index5 = list3.index('MPG')
                    MPG.append(list4[index5])
                else:
                    MPG.append('_')
                index6 = list3.index('Fuel type')
                Fuel_type.append(list4[index6])
                index7 = list3.index('Transmission')
                Transmission.append(list4[index7])
                index8 = list3.index('Stock #')
                Stock.append(list4[index8])
                index9 = list3.index('Mileage')
                Mileage.append(list4[index9])
                print(s+'YES')
                VIN.append(s)

In [ ]:
dict1={'VIN':VIN,
       'Car name':titles,
       'Prices':prices,
       'Exterior color':Exterior_color,
       'Interior color':Interior_color,
       'Engine':Engine,
       'Drivetrain':Drivetrain,
       'Transmission':Transmission,
       'Fuel type':Fuel_type,
       'Stock':Stock,
       'Mileage':Mileage}

df = pd.DataFrame.from_dict(dict1, orient="index")
df = df.transpose()
df.to_csv("cars.csv")